In [1]:
import pandas as pd
import os

raw = pd.read_csv("data/raw/Fire Incidents Data.csv", low_memory=False)
df = raw.copy()
log = [("rows in raw file", 0, len(df))]

In [2]:
def step(label, keep):
    """Keep only rows where `keep` is True, and record how many rows were removed."""
    global df
    removed = int((~keep).sum())
    df = df[keep].copy()
    log.append((label, removed, len(df)))

In [3]:
step("incident type not 01 or 02", df["Final_Incident_Type"].str[:2].isin(["01", "02"]))

In [4]:
df["alarm"] = pd.to_datetime(df["TFS_Alarm_Time"], errors="coerce")
df["arrival"] = pd.to_datetime(df["TFS_Arrival_Time"], errors="coerce")
step("missing alarm time", df["alarm"].notna())
step("missing arrival time", df["arrival"].notna())

In [5]:
df["wait_s"] = (df["arrival"] - df["alarm"]).dt.total_seconds()
step("wait under 10 s (negative or implausible)", df["wait_s"] >= 10)
step("wait over 3600 s (1 hour)", df["wait_s"] <= 3600)

In [6]:
df["has_coords"] = df["Latitude"].between(43.4, 44.0) & df["Longitude"].between(-79.8, -78.9)
print("incidents without usable coordinates:", (~df["has_coords"]).sum())

incidents without usable coordinates: 2


In [7]:
df["station"] = df["Incident_Station_Area"].str.strip().replace("", pd.NA)
print("distinct stations:", df["station"].nunique(), "| missing:", df["station"].isna().sum())

distinct stations: 86 | missing: 2


In [8]:
df["year"] = df["alarm"].dt.year
df["hour"] = df["alarm"].dt.hour
df["is_weekend"] = (df["alarm"].dt.dayofweek >= 5).astype(int)
df["daypart"] = pd.cut(df["hour"], [-1, 5, 11, 17, 23],
                       labels=["night", "morning", "afternoon", "evening"])

/var/folders/98/p89mc3nd1d34f9jndy3v2lz00000gn/T/ipykernel_33324/2229697215.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["is_weekend"] = (df["alarm"].dt.dayofweek >= 5).astype(int)


In [9]:
audit = pd.DataFrame(log, columns=["stage", "rows_removed", "rows_remaining"])
audit

,stage,rows_removed,rows_remaining
0,rows in raw file,0,36564
1,incident type not 01 or 02,10384,26180
2,missing alarm time,0,26180
3,missing arrival time,1,26179
4,wait under 10 s (negative or implausible),6,26173
5,wait over 3600 s (1 hour),5,26168


In [10]:
df["wait_s"].describe().round(0)

count    26168.0
mean       315.0
std        108.0
min         11.0
25%        251.0
50%        304.0
75%        366.0
max       1736.0
Name: wait_s, dtype: float64

In [11]:
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/incidents_clean.csv", index=False)
audit.to_csv("data/processed/cleaning_audit.csv", index=False)
print("saved", len(df), "rows")

saved 26168 rows


# Stage 2 summary: cleaning the incidents data

**Result:** 36,564 raw rows became 26,168 clean incidents. Every removal is recorded in cleaning_audit.csv.

| Step | Rows removed | Rows remaining |
|---|---|---|
| Raw file | 0 | 36,564 |
| Incident type not 01 or 02 | 10,384 | 26,180 |
| Missing alarm time | 0 | 26,180 |
| Missing arrival time | 1 | 26,179 |
| Wait under 10 s | 6 | 26,173 |
| Wait over 3,600 s | 5 | 26,168 |

## What the cleaning changed
- The median wait barely moved (309 s to 304 s), but the mean fell from 325 s to 315 s, the standard deviation from 262 s to 108 s, and the maximum from 40,942 s to 1,736 s.
- Apart from the type filter, only 12 rows were removed, so the cleaning did not reshape the data.

## Other decisions
- Station codes were trimmed of extra spaces (86 distinct codes remain, including one odd code, 145P, to check later).
- 2 incidents have no usable coordinates. They are kept and flagged (has_coords = False) so I can test later whether missing locations bias the results.
- Helper columns were added: year, hour, is_weekend and daypart.
- Cleaned data saved to data/processed/incidents_clean.csv.

## Some Important Points
1. Removing the impossible waits barely changed the median but sharply reduced the mean, the spread and the maximum, because a few extreme values pull those around while the median only counts the middle.
2. Missing-coordinate rows are flagged, not deleted, because deleting them could bias the results if they are not random, and flagging lets me test that later.